In [1]:
from typing import TypedDict, List, Optional, Annotated
import operator

# 2. Le cœur de LangGraph (Construction du graphe et routage)
from langgraph.graph import StateGraph, START, END

# 3. Gestion de la mémoire (Indispensable pour la pause "Human-in-the-loop")
from langgraph.checkpoint.memory import MemorySaver

# 4. (Optionnel mais recommandé) Les messages LangChain si vos nœuds utilisent directement le LLM
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, AnyMessage

In [2]:
from MedicalAgentState import MedicalAgentState

# 1. Extraction SEG & Calculs actuels

In [3]:
from etape_1 import node_process_current

# 2. Recherche dans l'Excel

In [4]:
from fetch_history import fetch_history

# 2 bis : extraction des résultats de computer vision sur les 3 dernieres studies.

In [5]:
from access_orthanc import node_process_orthanc

# 2 ter : comparaison 

In [6]:
from comparing import node_clinical_comparison

# 3. Draft report

In [7]:
from RAG import prepare_documents
from agent_rapport import RAGAgent, draft_report

# 4. Review report

In [8]:
from agent_review import review_report

# 5. Human validation

In [9]:
from savereport import save_report

# 6. Export final

In [10]:
def route_after_review(state: MedicalAgentState) -> str:
    """Si le relecteur trouve des erreurs ET qu'on a fait moins de 3 essais, on recommence."""
    feedback = state.get("critic_feedback", "")
    retries = state.get("revision_count", 0)
    
    if feedback != "OK" and retries < 3:
        print(f"  ⚠️ Erreur trouvée par le critique : {feedback}. Retour à la rédaction.")
        return "draft_report"
    
    # Si c'est OK ou qu'on a atteint la limite, on avance vers la validation finale
    return "save_report"

def route_after_history(state: MedicalAgentState) -> str:
    """
    Vérifie dans la mémoire si l'Excel a trouvé des antécédents.
    Si oui -> On télécharge les vieilles images sur Orthanc.
    Si non -> On passe directement à la rédaction du rapport.
    """
    historique_excel = state.get("patient_history", "")
    
    # Si la liste est vide (ou n'existe pas)
    if not historique_excel:
        print("  ⏭️ ROUTAGE : Aucun antécédent dans l'Excel. Saut d'Orthanc et de la comparaison.")
        return "draft_report"
    else:
        print("  🔀 ROUTAGE : Antécédents trouvés dans l'Excel. Envoi vers Orthanc (PACS).")
        return "access_orthanc"

workflow = StateGraph(MedicalAgentState)

# Ajout des nœuds
workflow.add_node("process_current", node_process_current)
workflow.add_node("fetch_history", fetch_history)
workflow.add_node("access_orthanc", node_process_orthanc)
workflow.add_node("comparison", node_clinical_comparison)
workflow.add_node("draft_report", draft_report)
workflow.add_node("review_report", review_report)
workflow.add_node("save_report",save_report)

# Définition du flux principal (les arêtes)
workflow.add_edge(START, "process_current")
workflow.add_edge("process_current", "fetch_history")

workflow.add_conditional_edges(
    "fetch_history",
    route_after_history,
    {
        "draft_report" :"draft_report",
        "access_orthanc" : "access_orthanc"
    }
)

workflow.add_edge("access_orthanc","comparison")
workflow.add_edge("comparison","draft_report")
workflow.add_edge("draft_report", "review_report")

workflow.add_conditional_edges(
    "review_report",
    route_after_review,
    {
        "draft_report": "draft_report",   # Retour en arrière
        "save_report": "save_report"    # On avance
    }
)
workflow.add_edge("save_report", END)

# ==========================================
# 5. COMPILATION AVEC PAUSE HUMAINE (Interrupt)
# ==========================================
# Le MemorySaver est vital pour mettre en pause et reprendre plus tard
medical_agent = workflow.compile()


if __name__ == "__main__":
    # 2. On définit l'étude que l'on veut analyser
    etude_cible = "1fc1a205-400007b2-d2a78317-dc63d295-6eb6bee9"
    
    # 3. On initialise l'état de départ
    inputs = {
        "current_study_id": etude_cible, 
        "excel_path": "data/protected-clinical-data.xlsx", 
        "excel_password": "Trois petits cochons",
        "history_texts": [],
        "revision_count": 0,
        "is_human_approved": True # On met à True dès le début puisqu'il n'y a plus d'humain
    }
    
    print("\n▶️ --- Lancement du flux IA 100% Automatisé ---")
    
    # 4. Utilisation de invoke() au lieu de stream() 
    # invoke() exécute tout le graphe d'un coup et renvoie l'état tout à la fin
    final_state = medical_agent.invoke(inputs)
    
    print("\n✅ Processus terminé. Rapport sauvegardé.")
    
    # 5. Affichage des résultats finaux stockés dans final_state
    print("\n" + "="*50)
    print("📋 RÉSUMÉ DU DOSSIER")
    print("="*50)
    print(f"Date : {final_state.get('study_date')} Patient       : {final_state.get('patient_id')} (Âge: {final_state.get('patient_age')}, Sexe: {final_state.get('patient_sex')})")
    print(f"Anomalie(s)   : {final_state.get('anomaly_detected')}")
    print("\n🤖 Avis du relecteur :", final_state.get("critic_feedback")) 
    
    print("\n📝 --- RAPPORT FINAL GÉNÉRÉ ---")
    print(final_state.get("draft_report"))


▶️ --- Lancement du flux IA 100% Automatisé ---
  ⬇️  Download of study n° 1fc1a205-400…
  ✅ Saved : studies/study_1fc1a205.zip  (51.2 Mo)
🔄 Analysis of the current study 0301B7D6...
  ⬇️  Download of study n° 1fc1a205-400…
  ✅ Saved : studies/study_1fc1a205.zip  (51.2 Mo)
[DEBUG] PatientID courant : 0301B7D6 | Date examen courant : 2020-07-14
[DEBUG] Examens trouvés pour patient dans l'historique : 0
  ⏭️ ROUTAGE : Aucun antécédent dans l'Excel. Saut d'Orthanc et de la comparaison.
[PDF] Manuscript_IRECIST_Lancet-Oncology_Seymour-et-al_revision_FINAL_clean_nov25.pdf → 67 chunks
[PDF] lung-rads-assessment-categories.pdf → 15 chunks
[PDF] RECISTGuidelines.pdf → 123 chunks

Total chunks préparés : 205
  ⚠️ Erreur trouvée par le critique : []. Retour à la rédaction.
[PDF] Manuscript_IRECIST_Lancet-Oncology_Seymour-et-al_revision_FINAL_clean_nov25.pdf → 67 chunks
[PDF] lung-rads-assessment-categories.pdf → 15 chunks
[PDF] RECISTGuidelines.pdf → 123 chunks

Total chunks préparés : 205
  ⚠️